In [1]:

!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import findspark
findspark.init()

In [2]:
import warnings
warnings.filterwarnings('ignore')
spark_ui_port = 4040
app_name = "Otus"

In [3]:
import pyspark
from pyspark import SparkContext
from pyspark.sql.types import StringType, DoubleType, IntegerType, StructType, StructField
import pyspark.sql.functions as F
from pyspark.sql.functions import isnan, when, count, col

spark = (
    pyspark.sql.SparkSession
        .builder
        .appName(app_name)
        .config("spark.executor.memory", "8g")
        .config("spark.driver.memory", "4g")
        .config("spark.executor.cores", 4)
        .config("spark.executor.instances", 3)
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter
spark.conf.set("hadoop.fs.defaultFS", "hdfs:/rc1d-dataproc-m-vnsfqf9g8b798so9.mdb.yandexcloud.net:8888/")

In [4]:
schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("tx_datetime", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("terminal_id", IntegerType()),
    StructField("tx_amount", DoubleType()),
    StructField("tx_time_seconds", IntegerType()),
    StructField("tx_time_days", IntegerType()),
    StructField("tx_fraud", IntegerType()),
    StructField("tx_fraud_scenario", IntegerType())
])

In [5]:
df = spark.read.csv('/user/ubuntu/data/2019-08-22.txt', schema=schema, header=True)

In [ ]:
df.show(5)

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|2019-08-22 06:51:03|          0|        711|    70.91|          24663|           0|       0|                0|
|             1|2019-08-22 05:10:37|          0|          0|    90.55|          18637|           0|       0|                0|
|             2|2019-08-22 19:05:33|          0|        753|    35.38|          68733|           0|       0|                0|
|             3|2019-08-22 07:21:33|          0|          0|    80.41|          26493|           0|       0|                0|
|             4|2019-08-22 09:06:17|          1|        981|   102.83|          32777|           0|       0|   

In [7]:
spark.read.csv('/user/ubuntu/data/2019-08-22.txt', schema=schema, header=True)\
.repartition(1).write.mode('overwrite').parquet('/user/ubuntu/data/dq_data')

In [5]:
df = spark.read.parquet('/user/ubuntu/data/dq_data')

In [6]:
df.show(5)

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|2019-08-22 06:51:03|          0|        711|    70.91|          24663|           0|       0|                0|
|             1|2019-08-22 05:10:37|          0|          0|    90.55|          18637|           0|       0|                0|
|             2|2019-08-22 19:05:33|          0|        753|    35.38|          68733|           0|       0|                0|
|             3|2019-08-22 07:21:33|          0|          0|    80.41|          26493|           0|       0|                0|
|             4|2019-08-22 09:06:17|          1|        981|   102.83|          32777|           0|       0|   

In [10]:
df.count()

46988418

In [ ]:
# 1. Проверка на наличие пустых значений
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|          0|          0|          0|        0|              0|           0|       0|                0|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+



In [15]:
# 2. Проверка на дубликаты
duplicate_count = df.count() - df.distinct().count()
print(f"Количество дубликатов: {duplicate_count}")

Количество дубликатов: 181


In [ ]:
# 3. Проверка на уникальность значений в определенных столбцах
# Пример: проверка уникальности значений в столбце 'transaction_id'
unique_id_count = df.select("transaction_id").distinct().count()
total_id_count = df.count()
print(f"Количество уникальных 'transaction_id': {unique_id_count}, Общее количество 'transaction_id': {total_id_count}, разница: {total_id_count - unique_id_count}")

Количество уникальных 'transaction_id': 46988237, Общее количество 'transaction_id': 46988418, разница: 181


In [ ]:
# 4. Анализ мин, макс, среднее
df.describe()

summary,transaction_id,tx_datetime,customer_id,terminal_id,tx_amount,tx_time_seconds,tx_time_days,tx_fraud,tx_fraud_scenario
count,46988418,46988418,46988418,46988418,46988418,46988418,46988418,46988418,46988418
mean,2.3494115552178305E7,null,500433.83151635365,26597.231571533222,54.2339599945678,1296054.5003670265,14.5006346883183,0.05377931642644364,0.10841507794537794
stddev,1.3564333711805945E7,null,288539.17376851145,1528137.6620335062,41.25033514383532,748049.5087349637,8.655415884394372,0.2255816983580882,0.45687805000298426
min,0,2019-08-22 00:00:00,-999999,0,0.0,0,0,0,0
max,9999999,2019-09-20 24:00:00,999999,89518096,3773.34,2592000,29,1,3


In [15]:
# 5. Фильтруем данные 
df_filtered = df.filter(col("customer_id") > 0).dropDuplicates()

In [16]:
df_filtered.show(5)

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|      35234342|2019-09-13 11:06:22|     497305|        344|    65.32|        1940782|          22|       0|                0|
|      35234713|2019-09-13 16:24:25|     497563|        731|    69.97|        1959865|          22|       0|                0|
|      35234777|2019-09-13 22:28:26|     497599|        880|     24.7|        1981706|          22|       0|                0|
|      35234893|2019-09-13 10:39:18|     497671|        757|     45.0|        1939158|          22|       0|                0|
|      35235292|2019-09-13 19:15:26|     497926|        414|    51.17|        1970126|          22|       0|   

In [17]:
df_filtered.describe()

summary,transaction_id,tx_datetime,customer_id,terminal_id,tx_amount,tx_time_seconds,tx_time_days,tx_fraud,tx_fraud_scenario
count,46988107,46988107,46988107,46988107,46988107,46988107,46988107,46988107,46988107
mean,2.3494124431763828E7,null,500436.4675817223,26595.499181314113,54.233935538624664,1296054.943578574,14.500639810835537,0.053779204171813096,0.10841483782268564
stddev,1.3564335530984363E7,null,288533.6526241682,1528086.9491070444,41.250217837414944,748049.6060933628,8.65541702637287,0.2255814763083082,0.4568775380575395
min,10,2019-08-22 00:00:00,1,0,0.0,0,0,0,0
max,9999999,2019-09-20 24:00:00,999999,89518096,3773.34,2592000,29,1,3


In [38]:
spark.stop()

In [ ]:
# from subprocess import Popen, PIPE
# hdfs_path = '/user/ubuntu/data'
# process = Popen(f'hdfs dfs -ls -h {hdfs_path}', shell=True, stdout=PIPE, stderr=PIPE)
# std_out, std_err = process.communicate()
# list_of_file_names = [fn.split(' ')[-1].split('/')[-1] for fn in std_out.decode().split('\n')[1:]][:-1]
# list_of_file_names_with_full_address = [fn.split(' ')[-1] for fn in std_out.decode().split('\n')[1:]][:-1]

In [ ]:
# list_of_file_names_with_full_address

['/user/ubuntu/data/2019-08-22.txt',
 '/user/ubuntu/data/2019-09-21.txt',
 '/user/ubuntu/data/2019-10-21.txt',
 '/user/ubuntu/data/2019-11-20.txt',
 '/user/ubuntu/data/2019-12-20.txt',
 '/user/ubuntu/data/2020-01-19.txt',
 '/user/ubuntu/data/2020-02-18.txt',
 '/user/ubuntu/data/2020-03-19.txt',
 '/user/ubuntu/data/2020-04-18.txt',
 '/user/ubuntu/data/2020-05-18.txt',
 '/user/ubuntu/data/2020-06-17.txt',
 '/user/ubuntu/data/2020-07-17.txt',
 '/user/ubuntu/data/2020-08-16.txt',
 '/user/ubuntu/data/2020-09-15.txt',
 '/user/ubuntu/data/2020-10-15.txt',
 '/user/ubuntu/data/2020-11-14.txt',
 '/user/ubuntu/data/2020-12-14.txt',
 '/user/ubuntu/data/2021-01-13.txt',
 '/user/ubuntu/data/2021-02-12.txt',
 '/user/ubuntu/data/2021-03-14.txt',
 '/user/ubuntu/data/2021-04-13.txt',
 '/user/ubuntu/data/2021-05-13.txt',
 '/user/ubuntu/data/2021-06-12.txt',
 '/user/ubuntu/data/2021-07-12.txt',
 '/user/ubuntu/data/2021-08-11.txt',
 '/user/ubuntu/data/2021-09-10.txt',
 '/user/ubuntu/data/2021-10-10.txt',
 

In [ ]:
# spark.read.csv(list_of_file_names_with_full_address[0], schema=schema, header=True)\
# .withColumn('file_name', F.lit(f'{list_of_file_names_with_full_address[0][-14:-4]}'))\
# .repartition(1).write\
# .partitionBy('file_name')\
# .mode('overwrite').parquet('/user/ubuntu/data/dq_data')
# for file in list_of_file_names_with_full_address[1:-1]:
#     spark.read.csv(f'{file}', schema=schema, header=True)\
#     .withColumn('file_name', F.lit(f'{file[-14:-4]}'))\
#     .repartition(1).write\
#     .partitionBy('file_name')\
#     .mode('append').parquet('/user/ubuntu/data/dq_data')
#     call(f'hdfs dfs -rm -r -skipTrash {file}', shell=True)